# Goal 1.5 전체 계층형 합성 모델 재현

## 목표
기존 1차 모델을 새 학습 없이 합성 validation 샘플에 적용합니다. 결과 범위는 `oracle/sanity`, 실제 정확도는 `NOT VERIFIED`입니다.

## 설정

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import tarfile

RUN_TRAINING = False
RUN_LOCKED_TEST = False
USE_GPU = False
MODEL_ROOT_OVERRIDE = None
DATASET_ROOT_OVERRIDE = None
assert not RUN_TRAINING and not RUN_LOCKED_TEST and not USE_GPU
print({
    'run_training': RUN_TRAINING,
    'run_locked_test': RUN_LOCKED_TEST,
    'use_gpu': USE_GPU,
})

## 입력 탐색

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if MODEL_ROOT_OVERRIDE:
    model_root = Path(MODEL_ROOT_OVERRIDE)
else:
    manifests = sorted(Path('/kaggle/input').rglob('model_manifest.json'))
    if len(manifests) != 1:
        raise RuntimeError(f'Kaggle Model manifest must be unique: {manifests}')
    model_root = manifests[0].parent
if DATASET_ROOT_OVERRIDE:
    dataset_root = Path(DATASET_ROOT_OVERRIDE)
    dataset_manifests = [
        dataset_root / 'manifest.json',
        dataset_root / 'prepared__manifest.json',
    ]
    dataset_manifest = next(
        (path for path in dataset_manifests if path.is_file()), None
    )
    if dataset_manifest is None:
        raise RuntimeError('prepared Dataset manifest is missing')
else:
    candidates = []
    manifest_paths = list(Path('/kaggle/input').rglob('manifest.json'))
    manifest_paths += list(
        Path('/kaggle/input').rglob('prepared__manifest.json')
    )
    for path in manifest_paths:
        try:
            manifest = json.loads(path.read_text())
            if manifest.get('prepared_schema') == 'goal1.5/prepared/v1':
                candidates.append(path)
        except (json.JSONDecodeError, OSError):
            pass
    if len(candidates) != 1:
        message = f'prepared synthetic Dataset must be unique: {candidates}'
        raise RuntimeError(message)
    dataset_manifest = candidates[0]
    dataset_root = dataset_manifest.parent
print({'model_root': str(model_root), 'dataset_root': str(dataset_root)})

## 무결성 검증

In [ ]:
outer = json.loads((model_root / 'model_manifest.json').read_text())
archive = model_root / outer['archive_path']
payload_root = model_root / 'model_payload'
if not payload_root.is_dir():
    payload_root = model_root / 'extracted'
if not payload_root.is_dir():
    payload_root = Path('/kaggle/working/model_payload')
    payload_root.mkdir(parents=True, exist_ok=False)
    with tarfile.open(archive, 'r:gz') as bundle:
        unsafe = any(
            member.name.startswith('/') or '..' in Path(member.name).parts
            for member in bundle.getmembers()
        )
        if unsafe:
            raise RuntimeError('unsafe archive member')
        bundle.extractall(payload_root, filter='data')
wheelhouse = payload_root / 'wheel'
wheels = sorted(wheelhouse.glob('*.whl'))
required_modules = ('multisensor_ml', 'skops')
working_root = Path('/kaggle/working')
if not working_root.is_dir():
    working_root = Path.cwd()
runtime_root = working_root / 'multisensor_runtime'
if any(importlib.util.find_spec(name) is None for name in required_modules):
    runtime_root.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable, '-m', 'pip', 'install', '--target', str(runtime_root),
        '--no-index', '--no-deps',
        *map(str, wheels),
    ]
    subprocess.check_call(command)
print({'package_sha256': outer['archive_sha256'], 'runtime': str(runtime_root)})

## 모델 로딩과 계층형 추론

In [ ]:
comparison_path = working_root / 'comparison.json'
prediction_path = working_root / 'prediction.parquet'
inference_code = r'''
import json
import sys
from pathlib import Path
import pandas as frame_library
from multisensor_ml.kaggle_model_package import verify_kaggle_model_package
from multisensor_ml.kaggle_reproduce import (
    compare_expected, load_hierarchical_package, predict_hierarchical,
)
model_root = Path(sys.argv[1])
payload_root = Path(sys.argv[2])
comparison_path = Path(sys.argv[3])
prediction_path = Path(sys.argv[4])
verify_kaggle_model_package(model_root)
package = load_hierarchical_package(payload_root)
sample = frame_library.read_parquet(payload_root / 'sample/sample_input.parquet')
expected_path = payload_root / 'sample/expected_output.parquet'
expected = frame_library.read_parquet(expected_path)
prediction = predict_hierarchical(package, sample)
comparison = compare_expected(prediction, expected)
prediction.to_parquet(prediction_path, index=False)
comparison_path.write_text(json.dumps({
    'status': comparison.status,
    'compared_rows': comparison.compared_rows,
    'first_mismatch_column': comparison.first_mismatch_column,
    'first_mismatch_key': comparison.first_mismatch_key,
}, sort_keys=True) + '\n')
'''
inference_env = dict(os.environ)
existing_pythonpath = inference_env.get('PYTHONPATH', '')
inference_env['PYTHONPATH'] = str(runtime_root) + os.pathsep + existing_pythonpath
inference_command = [
    sys.executable, '-c', inference_code, str(model_root), str(payload_root),
    str(comparison_path), str(prediction_path),
]
subprocess.check_call(inference_command, env=inference_env)
comparison = json.loads(comparison_path.read_text())
comparison

## 재현 비교와 주요 결과

In [ ]:
receipt = {
    'status': comparison['status'],
    'compared_rows': comparison['compared_rows'],
    'first_mismatch_column': comparison['first_mismatch_column'],
    'model_package_sha256': outer['archive_sha256'],
    'dataset_root': str(dataset_root),
    'dataset_manifest_sha256': sha256_file(dataset_manifest),
    'source_dataset_handle': outer['source_dataset_handle'],
    'source_dataset_version': outer['source_dataset_version'],
    'device': 'cpu',
    'run_training': RUN_TRAINING,
    'run_locked_test': RUN_LOCKED_TEST,
    'data_status': 'oracle/sanity',
    'real_data_status': 'NOT VERIFIED',
}
receipt_text = json.dumps(receipt, indent=2, sort_keys=True) + '\n'
Path('reproduction_receipt.json').write_text(receipt_text)
print(json.dumps(receipt, ensure_ascii=False, sort_keys=True))
if comparison['status'] != 'REPRODUCED':
    raise RuntimeError(f"FAILED: {comparison['first_mismatch_column']}")

## 다음 아이디어

개인 기준선, STD-A, 사건, 5단계, 행동 확률을 함께 보되 행동은 보조지표로 해석합니다. 실제 센서 단계에서는 PostgreSQL/TimescaleDB 운영 저장계약과 원시 Parquet를 분리해 검증합니다.